[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/magnetic_materials/magnetic_materials.ipynb)

# Magnetic Materials

This example computes the shared magnetic field and axial currents across a permanent magnet, a soft-iron core, and an isolated copper conductor disturbed by a current pulse in a nearby coil. The cross-section is modelled as uniform along one spatial axis, so currents run perpendicular to the plane while magnetic field lines close within it; the iron reorients its reversible magnetization as the pulse passes, and induced currents in the copper delay the field's relaxation after the coil switches off.

The term ***simplex gradient*** here means the spatial slope across a triangle associated with one of its corners. Wedging those slopes with the axial potential direction gives the cross-sectional magnetic field `gradients ^ z`, which pairs with intrinsic excitation planes both to assemble the iron's permeability-weighted magnetic storage and to project the permanent magnet's remanence into a bound-current load. Sweeping the axial direction along the rest-frame worldline forms `axis & time`, so `electric & (axis & time)` reads the axial electric field from `((rate - mean_rate) * t) ^ z` and drives conduction 4-currents along `axis`.

In Gibbs–Heaviside vector notation, an axial vector potential $\mathbf{A} = A_z(x, y, t)\,\hat{\mathbf{z}}$ enters through rotated partial derivatives $\mathbf{B} = \nabla \times (A_z\hat{\mathbf{z}}) = (\partial_y A_z)\,\hat{\mathbf{x}} - (\partial_x A_z)\,\hat{\mathbf{y}}$, bound current requires the out-of-plane curl component $(\nabla \times \mathbf{M}_r)\cdot\hat{\mathbf{z}} = \partial_x M_{r,y} - \partial_y M_{r,x}$, and axial induction adds a separate scalar-potential gradient $E_z = -\partial_t A_z - \partial_z\phi$ constrained by $\int \sigma(-\partial_t A_z - \partial_z\phi)\,dx\,dy = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from collections.abc import Iterator
import numpy as np
from numga import Algebra, Extensor, NumpyContext, stack
from numga.sparse import Sparse, SparseExtensor, spdiag
from numga.topology import Topology
from examples.electromagnetism.magnetic_materials import render, scenarios
from examples.util.animation import save_figure, show_figure, show_states

ga = Algebra("x+y+z+t-w0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                   # Scalar+
Upright = ga.gatype.from_blades("x y")                        # Vector+
Point = ga.gatype.antivector()                                # Antivector-
Current = ga.gatype.antivector()                              # Antivector-
Field = ga.gatype.bivector()                                  # Bivector+
Excitation = ga.gatype.antibivector()                         # Antibivector-
origin = mv.w.dual()                                          # Point-
axis = mv.z.dual()                                            # Current-
time = -mv.t.dual()                                           # Current-
sweep = axis & time                                           # Excitation-
magnetic_planes = stack((mv.xz.dual(), mv.yz.dual()))         # [2] Excitation-
magnetic_response = (magnetic_planes * (Field & magnetic_planes)).sum(axis=0)  # Excitation- <- Field+
conduction = axis * (Field & sweep)                           # Current- <- Field+

## Materials in one cross-section

A permanently magnetized bar supplies a steady magnetic flux, a surrounding C-shaped iron core concentrates that flux, and a hollow copper cylinder sits in the return path beside two coil cross-sections carrying equal and opposite prescribed currents. Smooth transitions place all three material fields across one shared triangulated rectangle.

In [ ]:
n_across, n_up, n_frames = scenarios.n_across, scenarios.n_up, scenarios.n_frames
frame_step = scenarios.frame_step
topology, vertices, fractions, permeability, conductivity, remanence, coils, drive = scenarios.material_scene(
    n_across, n_up, n_frames,
)
setup_plot = render.setup_figure(topology, vertices, fractions, coils)
save_figure(setup_plot, "magnetic_materials_setup")
show_figure(setup_plot)

Light outlines mark the boundaries of the permanent magnet on the left, the C-shaped soft-iron core around it, and the copper ring in the gap, with the two circular coil cross-sections to the right.

## Magnetic storage and bound current

An axial potential `potential * z` has cross-sectional exterior derivative `(gradients ^ z) * (corners * potential)`, so curves of constant `potential` follow the magnetic field lines. Dividing `magnetic_response` by the local permeability gives the reluctance map whose pairing `field & reluctance(field) / 2` measures magnetic energy density in the iron and surrounding space, while pairing `gradients ^ z` with the bar's prescribed `remanence` sums to the bound-current load concentrated along the magnet's two flanks. Only the outer boundary holds `potential` fixed; the interior solve spans every material including the permanent magnet.

In [ ]:
def triangle_areas(topology: Topology, vertices: Point) -> Extensor:
    """Signed corner joins with `sweep` measure each triangle's cross-sectional area (`Scalar+[F]`)."""
    triangle_vertices = topology.incidence[2, 0] * vertices       # [3] Point-[F]
    double_areas = triangle_vertices[0] & triangle_vertices[1] & triangle_vertices[2] & sweep  # Antiscalar-[F]
    return double_areas.norm() / 2                                # Scalar+[F]


def simplex_gradients(topology: Topology, vertices: Point) -> Extensor:
    """Opposite edge joins determine the three corner simplex gradients on each triangle (`[3] Upright+[F]`)."""
    triangle_vertices = topology.incidence[2, 0] * vertices       # [3] Point-[F]
    facing = topology.off[2, 0] * vertices                        # [3, 2] Point-[F]
    sides = facing[0] & facing[1]                                 # [3] Excitation-[F]
    slopes = (sides & sweep) / (sides & sweep & triangle_vertices)  # [3] Vector+[F]
    return slopes.cast(Upright)                                   # [3] Upright+[F]


areas = triangle_areas(topology, vertices)                        # Scalar+[F]
dual_areas = topology.incidence[2, 0].mean(axis=0).adjugate() * areas  # Scalar+[V]
gradients = simplex_gradients(topology, vertices)                 # [3] Upright+[F]

In [ ]:
def magnetic_stiffness(topology: Topology, areas: Scalar, gradients: Upright,
                       permeability: Scalar) -> SparseExtensor:
    """Permeability weights the magnetic storage coupling across triangle corners (`Scalar+[V <- V]`)."""
    corners = topology.incidence[2, 0]                            # [3] Scalar+[F <- V]
    # An axial potential `potential * z` produces the cross-sectional magnetic field `gradients ^ z`.
    field_gradients = gradients ^ mv.z                            # [3] Field+[F]
    reluctance = magnetic_response / (corners.mean(axis=0) * permeability)  # (Excitation- <- Field+)[F]
    local_stiffness = areas * (field_gradients[:, None] & reluctance(field_gradients[None, :]))  # [3, 3] Scalar+[F]
    return (corners[:, None].adjugate() * spdiag(local_stiffness) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]


def remanent_source(topology: Topology, areas: Scalar, gradients: Upright,
                    remanence: Excitation) -> Extensor:
    """Permanent magnetization supplies an integrated bound-current load at each vertex (`Scalar+[V]`)."""
    corners = topology.incidence[2, 0]                            # [3] Scalar+[F <- V]
    field_gradients = gradients ^ mv.z                            # [3] Field+[F]
    triangle_remanence = corners.mean(axis=0) * remanence         # Excitation-[F]
    return (corners.adjugate() * (areas * (field_gradients & triangle_remanence))).sum(axis=0)  # Scalar+[V]


stiffness = magnetic_stiffness(topology, areas, gradients, permeability)  # Scalar+[V <- V]
source = remanent_source(topology, areas, gradients, remanence)   # Scalar+[V]
extension = topology.reading(topology.interior[0], 0).adjugate()  # Scalar+[V <- V_interior]
coil_load = dual_areas * (coils / (dual_areas * coils).sites.sum() * scenarios.coil_signs).sum(axis=0)  # Scalar+[V]

## Induced currents under a coil pulse

A time-varying axial potential generates an electric field `((rate - mean_rate) * t) ^ z`, which `conduction` maps into an axial 4-current whose magnetic field opposes the change. Because the isolated copper ring has no external leads, rapid charge redistribution subtracts the conductance-weighted `mean_rate` so inward and outward axial currents across the ring balance while counter-flowing eddy currents still circulate across opposite sides of the wall. The midpoint step couples magnetic storage to this resistive response as the coil opposes part of the permanent magnet's flux and then switches off. Computing and rendering the animation in the next cell takes a few seconds.

In [ ]:
def evolution(stiffness: SparseExtensor, source: Scalar, dual_areas: Scalar,
              conductivity: Scalar, extension: Sparse, coil_load: Scalar,
              drive: np.ndarray, step: float) -> Iterator[tuple[Extensor, Extensor]]:
    """Yields `(potential, current)` fields (`Scalar+[V]`, `Current-[V]`) across a coil pulse."""
    conductance_mass = dual_areas * conductivity                  # Scalar+[V]
    bound_current = axis * (source / dual_areas)                  # Current-[V]
    coil_current = axis * (coil_load / dual_areas)                # Current-[V]
    initial = (extension.adjugate() * stiffness * extension) * Scalar  # Scalar+[V_interior <- V_interior]
    potential = extension * initial.solve(extension.adjugate() * (source + coil_load * drive[0]))  # Scalar+[V]
    yield potential, bound_current + drive[0] * coil_current
    storage = spdiag(conductance_mass / step)                     # Scalar+[V <- V]
    implicit = (extension.adjugate() * (storage + stiffness / 2) * extension) * Scalar  # Scalar+[V_interior <- V_interior]
    explicit = storage - stiffness / 2                            # Scalar+[V <- V]
    # An isolated conductor redistributes charge until its net axial current vanishes.
    uniform = extension * implicit.solve(extension.adjugate() * (conductance_mass / step))  # Scalar+[V]
    total_conductance = conductance_mass.sites.sum()              # Scalar+
    normalization = 1 - (conductance_mass * uniform).sites.sum() / total_conductance  # Scalar+
    for previous_drive, following_drive in zip(drive[:-1], drive[1:]):
        midpoint_load = source + coil_load * ((previous_drive + following_drive) / 2)  # Scalar+[V]
        mean_potential = (conductance_mass * potential).sites.sum() / total_conductance  # Scalar+
        particular = extension * implicit.solve(
            extension.adjugate() * (explicit * potential + midpoint_load - conductance_mass * mean_potential / step)
        )                                                         # Scalar+[V]
        mean_following = (conductance_mass * particular).sites.sum() / total_conductance / normalization  # Scalar+
        following = particular + uniform * mean_following         # Scalar+[V]
        rate = (following - potential) / step                     # Scalar+[V]
        mean_rate = (conductance_mass * rate).sites.sum() / total_conductance  # Scalar+
        electric = ((rate - mean_rate) * mv.t) ^ mv.z             # Field+[V]
        induced_current = conductivity * conduction(electric)     # Current-[V]
        potential = following                                     # Scalar+[V]
        yield potential, bound_current + following_drive * coil_current + induced_current


states = show_states(
    evolution(stiffness, source, dual_areas, conductivity, extension, coil_load, drive, frame_step),
    render.field_animation, topology, vertices, fractions, coils,
    name="magnetic_materials_notebook", duration_ms=scenarios.frame_duration_ms,
    n_frames=n_frames, colors=128,
)

Gold contours trace lines of magnetic flux through the magnet, iron core, copper ring and surrounding air, while red and blue shade positive and negative axial 4-current density `z & current` across the magnet's flanks, the two coil wires and the copper wall on one shared scale.

## Resistive dissipation

Pairing the axial electric field with the worldsheet `induced_currents & time` gives the resistive power density dissipated in the copper. Heating peaks while the coil current rises and falls, and continues after the dotted pulse-end marker while the trapped eddy currents decay.

In [ ]:
def joule_heating(potentials: Scalar, dual_areas: Scalar, conductivity: Scalar,
                  step: float) -> Extensor:
    """Resistive power integrates `electric & (induced_currents & time)` across the conductor (`[...] Scalar+`)."""
    conductance_mass = dual_areas * conductivity                  # Scalar+[V]
    rates = (potentials[1:] - potentials[:-1]) / step             # [...] Scalar+[V]
    mean_rates = (conductance_mass * rates).sites.sum() / conductance_mass.sites.sum()  # [...] Scalar+
    electric = ((rates - mean_rates) * mv.t) ^ mv.z               # [...] Field+[V]
    induced_currents = conductivity * conduction(electric)        # [...] Current-[V]
    return (dual_areas * (electric & (induced_currents & time))).sites.sum()  # [...] Scalar+


potentials = stack([potential for potential, _ in states])        # [n_frames] Scalar+[V]
currents = stack([current for _, current in states])              # [n_frames] Current-[V]
heating = joule_heating(potentials, dual_areas, conductivity, frame_step)  # [n_frames - 1] Scalar+
relaxation_plot = render.relaxation_figure(heating, frame_step, scenarios.pulse_fall)
save_figure(relaxation_plot, "magnetic_materials_notebook_relaxation")
show_figure(relaxation_plot)

The orange curve shows total Joule power over time, with the vertical dotted line marking the moment the prescribed coil current returns to zero.

In [ ]:
# checks
assert potentials.isfinite().all()
assert (gradients.sum(axis=0).norm() < 1e-8).all()
assert source.sites.sum().abs() < 1e-8
assert ((dual_areas * (mv.z & currents)).sites.sum().abs() < 1e-8).all()
assert (heating >= 0).all()
assert heating[-1] > 0